In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import pairwise_distances
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import copy

In [2]:
artnet_2025= pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe.xlsx")

In [3]:
artnet_2025_embed = np.load("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\clip_embedding.npy",allow_pickle=True)

In [9]:
print(artnet_2025_embed.nbytes/1024/1024, "MB")

726.12890625 MB


In [30]:
artnet_2025_embed.shape

(371778, 512)

artnet_2025_embed1 = np.array([x.squeeze() for x in artnet_2025_embed])
np.save("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\clip_embedding.npy", artnet_2025_embed1)

In [9]:
artnet_2025.shape

(371778, 26)

In [11]:
artnet_2025.columns

Index(['lot id', 'artwork id', 'artist id', 'sale date', 'artist modifier',
       'first', 'last', 'nationality', 'year born', 'year died', 'title',
       'workyear modifier', 'workyear from', 'workyear to', 'est lo', 'est hi',
       'sale price', 'currency', 'currency exchange rate', 'est lo usd',
       'est hi usd', 'sale price usd', 'priceStatus', 'pricePhrase',
       'has_image', 'sale year'],
      dtype='object')

In [15]:
artnet_2025[artnet_2025["workyear from"] == 1998].index

Index([366671, 366672, 366673, 366674, 366675, 366676, 366677, 366678, 366679,
       366680,
       ...
       367481, 367482, 367483, 367484, 367485, 367486, 367487, 367488, 367489,
       367490],
      dtype='int64', length=820)

In [4]:
year_to_prior = {}
current_prior = set()
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start Building Prior Knowledge")
for year in sorted(artnet_2025["workyear from"].unique()):
    year_rows_index = artnet_2025[artnet_2025["workyear from"] == year].index
    year_to_prior[year] = copy.deepcopy(current_prior)  # store snapshot before update
    current_prior |= set(year_rows_index)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Finish Building Prior Knowledge")

2025-10-30 22:50:32: Start Building Prior Knowledge
2025-10-30 22:50:33: Finish Building Prior Knowledge


In [5]:
def compute_newness(i,row_dict, year_to_prior,embeddings,BASELINE_YEAR):
    year = row_dict["workyear from"]
    prior_knowledge = embeddings[list(year_to_prior.get(year, set()))]
    A = embeddings[i]
    if year <= BASELINE_YEAR:  # pre-year baseline
        return i, 0
    # print(f"The total steps are: {playcount}")
    # mark new pairs not in prior knowledge
    distances = np.linalg.norm(prior_knowledge - A, axis=1)
    min_dist = np.min(distances)
    return i, min_dist

In [8]:
# number_size=artnet_2025.shape[0]
# range_start = 30000
range_start = 200000
number_size = 100000
BASELINE_YEAR=1600
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)

In [9]:
newness = np.zeros(N)
max_workers = 10  # tune to your CPU cores
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Parallel computation starts")
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {
        ex.submit(compute_newness, i, 
                artnet_2025.iloc[i],
                year_to_prior,
                artnet_2025_embed,
                BASELINE_YEAR): i
        for i in range(range_start,range_end)
    }
    
    completed = 0
    for fut in as_completed(futures):
        i,row_newness = fut.result()
        newness[i-range_start]  = row_newness
        completed += 1
        if completed % 10000 == 0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{artnet_2025.shape[0]}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Newness\\newness_{range_start}.npy", newness)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")

2025-10-31 00:12:04: Parallel computation starts
2025-10-31 00:21:28: Completed 10000/371778
2025-10-31 00:31:33: Completed 20000/371778
2025-10-31 00:41:48: Completed 30000/371778
2025-10-31 00:52:45: Completed 40000/371778
2025-10-31 01:04:32: Completed 50000/371778
2025-10-31 01:17:13: Completed 60000/371778
2025-10-31 01:30:32: Completed 70000/371778
2025-10-31 01:44:40: Completed 80000/371778
2025-10-31 01:59:33: Completed 90000/371778
2025-10-31 02:15:16: Completed 100000/371778
2025-10-31 02:15:16: Ends
2025-10-31 02:15:16: Saving embeddings
2025-10-31 02:15:16: Embedding Saved


In [10]:
range_start = 200000
number_size = 100000
BASELINE_YEAR=1600
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)

In [11]:
newness = np.zeros(N)
max_workers = 10  # tune to your CPU cores
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Parallel computation starts")
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {
        ex.submit(compute_newness, i, 
                artnet_2025.iloc[i],
                year_to_prior,
                artnet_2025_embed,
                BASELINE_YEAR): i
        for i in range(range_start,range_end)
    }
    
    completed = 0
    for fut in as_completed(futures):
        i,row_newness = fut.result()
        newness[i-range_start]  = row_newness
        completed += 1
        if completed % 10000 == 0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{artnet_2025.shape[0]}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Newness\\newness_{range_start}.npy", newness)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")

2025-10-31 02:15:16: Parallel computation starts
2025-10-31 02:31:52: Completed 10000/371778
2025-10-31 02:49:11: Completed 20000/371778
2025-10-31 03:07:18: Completed 30000/371778
2025-10-31 03:26:10: Completed 40000/371778
2025-10-31 03:45:52: Completed 50000/371778
2025-10-31 04:06:26: Completed 60000/371778
2025-10-31 04:27:51: Completed 70000/371778
2025-10-31 04:49:56: Completed 80000/371778
2025-10-31 05:12:55: Completed 90000/371778
2025-10-31 05:36:40: Completed 100000/371778
2025-10-31 05:36:40: Ends
2025-10-31 05:36:40: Saving embeddings
2025-10-31 05:36:40: Embedding Saved


In [12]:
range_start = 300000
number_size = 100000
BASELINE_YEAR=1600
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)

In [13]:
newness = np.zeros(N)
max_workers = 10  # tune to your CPU cores
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Parallel computation starts")
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {
        ex.submit(compute_newness, i, 
                artnet_2025.iloc[i],
                year_to_prior,
                artnet_2025_embed,
                BASELINE_YEAR): i
        for i in range(range_start,range_end)
    }
    
    completed = 0
    for fut in as_completed(futures):
        i,row_newness = fut.result()
        newness[i-range_start]  = row_newness
        completed += 1
        if completed % 10000 == 0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{artnet_2025.shape[0]}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Newness\\newness_{range_start}.npy", newness)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")

2025-10-31 05:36:40: Parallel computation starts
2025-10-31 06:01:25: Completed 10000/371778
2025-10-31 06:26:59: Completed 20000/371778
2025-10-31 06:53:37: Completed 30000/371778
2025-10-31 07:21:03: Completed 40000/371778
2025-10-31 07:49:22: Completed 50000/371778
2025-10-31 08:18:26: Completed 60000/371778
2025-10-31 08:48:27: Completed 70000/371778
2025-10-31 08:53:50: Ends
2025-10-31 08:53:50: Saving embeddings
2025-10-31 08:53:50: Embedding Saved


In [ ]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)
max_workers = 20
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2025.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    newness = np.zeros(N)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(compute_newness, i, 
                    artnet_2025.iloc[i],
                    year_to_prior,
                    artnet_2025_embed,
                    BASELINE_YEAR): i
            for i in range(range_start,range_end)
        }
        
        completed = 0
        for fut in as_completed(futures):
            i,row_newness = fut.result()
            newness[i-range_start]  = row_newness
            completed += 1
            if completed % 10000 == 0:
                print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{artnet_2025.shape[0]}")
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Newness\\newness_{range_start}.npy", newness)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2025.shape[0])
    N = min(number_size, artnet_2025.shape[0]-range_start)
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-10-30 19:21:33: Start
2025-10-30 19:21:33: Now at 0
2025-10-30 19:22:03: Completed 10000/371778
2025-10-30 19:23:13: Completed 20000/371778


# Formal

In [2]:
path = "D:\\Research"

In [3]:
artnet_2024 = pd.read_excel(f"{path}\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [4]:
artnet_2024.shape

(355551, 16)

In [5]:
artnet_2024_embed = np.load(f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Embedding_2024.npy",allow_pickle=True)

artnet_2024_embed = np.asarray(artnet_2024_embed, dtype=np.float32)

np.save(f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Embedding_2024.npy", artnet_2024_embed)

In [6]:
artnet_2024_embed.shape

(355551, 768)

In [7]:
year_to_prior = {}
current_prior = set()
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start Building Prior Knowledge")
for year in sorted(artnet_2024["workyear from"].unique()):
    year_rows_index = artnet_2024[artnet_2024["workyear from"] == year].index
    year_to_prior[year] = copy.deepcopy(current_prior)  # store snapshot before update
    current_prior |= set(year_rows_index)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Finish Building Prior Knowledge")

2026-01-03 18:15:29: Start Building Prior Knowledge
2026-01-03 18:15:46: Finish Building Prior Knowledge


In [8]:
def compute_newness(i,row_dict, year_to_prior,embeddings,BASELINE_YEAR):
    year = row_dict["workyear from"]
    prior_knowledge = embeddings[list(year_to_prior.get(year, set()))]
    A = embeddings[i]
    if year <= BASELINE_YEAR:  # pre-year baseline
        return i, 0
    # print(f"The total steps are: {playcount}")
    # mark new pairs not in prior knowledge
    distances = np.linalg.norm(prior_knowledge - A, axis=1)
    min_dist = np.min(distances)
    return i, min_dist

def compute_newness(i,year,prior_knowledge,A,BASELINE_YEAR):
    if year <= BASELINE_YEAR:  # pre-year baseline
        del A,year,prior_knowledge
        return i, 0
    # print(f"The total steps are: {playcount}")
    # mark new pairs not in prior knowledge
    distances = np.linalg.norm(prior_knowledge - A, axis=1)
    min_dist = np.min(distances)
    del A,year,prior_knowledge
    return i, min_dist

number_size = 10000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)
max_workers = 10
BASELINE_YEAR=1600
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2024.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    newness = np.zeros(N)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(compute_newness, i, 
                    artnet_2024.iloc[i]["workyear from"],
                    artnet_2024_embed[list(year_to_prior.get(artnet_2024.iloc[i]["workyear from"], set()))],
                    artnet_2024_embed[i],
                    BASELINE_YEAR): i
            for i in range(range_start,range_end)
        }
        completed = 0
        for fut in as_completed(futures):
            i,row_newness = fut.result()
            newness[i-range_start]  = row_newness
            completed += 1
            if completed % 1000 == 0:
                print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{artnet_2024.shape[0]}")
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    np.save(f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Newness_2024\\newness_{range_start}.npy", newness)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2024.shape[0])
    N = min(number_size, artnet_2024.shape[0]-range_start)
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

In [9]:
number_size = 10000
range_start = 130000
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)
max_workers = 10
BASELINE_YEAR=1600
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2024.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    newness = np.zeros(N)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(compute_newness, i, 
                    artnet_2024.iloc[i],
                    year_to_prior,
                    artnet_2024_embed,
                    BASELINE_YEAR): i
            for i in range(range_start,range_end)
        
        }
        completed = 0
        for fut in as_completed(futures):
            i,row_newness = fut.result()
            newness[i-range_start]  = row_newness
            completed += 1
            if completed % 1000 == 0:
                print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{artnet_2024.shape[0]}")
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    np.save(f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Newness_2024\\newness_{range_start}.npy", newness)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2024.shape[0])
    N = min(number_size, artnet_2024.shape[0]-range_start)
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2026-01-03 18:15:46: Start
2026-01-03 18:15:46: Now at 130000
2026-01-03 18:18:46: Completed 1000/355551
2026-01-03 18:22:11: Completed 2000/355551
2026-01-03 18:25:55: Completed 3000/355551
2026-01-03 18:29:39: Completed 4000/355551
2026-01-03 18:33:42: Completed 5000/355551
2026-01-03 18:38:23: Completed 6000/355551
2026-01-03 18:43:42: Completed 7000/355551
2026-01-03 18:48:10: Completed 8000/355551
2026-01-03 18:52:54: Completed 9000/355551
2026-01-03 18:57:44: Completed 10000/355551
2026-01-03 18:57:44: Saving embeddings
2026-01-03 18:57:44: Embedding Saved
2026-01-03 18:57:44: Now at 140000
2026-01-03 19:01:39: Completed 1000/355551
2026-01-03 19:05:54: Completed 2000/355551
2026-01-03 19:10:28: Completed 3000/355551
2026-01-03 19:14:48: Completed 4000/355551
2026-01-03 19:19:23: Completed 5000/355551
2026-01-03 19:24:14: Completed 6000/355551
2026-01-03 19:28:31: Completed 7000/355551
2026-01-03 19:33:33: Completed 8000/355551
2026-01-03 19:38:09: Completed 9000/355551
2026-01-0

# Merge

In [2]:
import pickle
import glob

In [3]:
path = "D:\\Research"

In [6]:
folder_path = f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Newness_2024"
npy_files = glob.glob(f"{folder_path}/*.npy")

In [7]:
len(npy_files)

36

In [8]:
arrays = [np.load(f, allow_pickle=True) for f in npy_files]
big_array = np.concatenate(arrays, axis=0)
np.save(f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Newness_2024.npy", big_array)

In [9]:
big_array

array([0.82489675, 0.89730227, 0.75852078, ..., 0.80119944, 0.4563618 ,
       0.62966174], shape=(355551,))

In [10]:
newness_array = np.vstack(big_array)

In [11]:
newness_array.shape

(355551, 1)

In [12]:
np.save(f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Newness_2024.npy", newness_array)